In [ ]:
import pandas as pd, numpy as np, requests, json, os, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay)
import joblib
from mplsoccer import Pitch
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)
os.makedirs('dashboard', exist_ok=True)

In [ ]:
BASE = "https://premier.72-60-245-2.sslip.io"

def load_or_download(path, url):
    if os.path.exists(path):
        print(f"[caché] Cargando {path}")
        return pd.read_csv(path)
    print(f"[descarga] {url}")
    df = pd.read_csv(url)
    df.to_csv(path, index=False)
    print(f"[guardado] {path} — {df.shape}")
    return df

players = load_or_download('data/players.csv', f"{BASE}/export/players")
matches = load_or_download('data/matches.csv', f"{BASE}/export/matches")
events  = load_or_download('data/events.csv',  f"{BASE}/export/events")

In [ ]:
for name, df in [('players', players), ('matches', matches), ('events', events)]:
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Nulos: {df.isnull().sum()[df.isnull().sum()>0].to_dict()}")
    display(df.head(2))

In [ ]:
if 'as_' in matches.columns:
    matches.rename(columns={'as_': 'away_shots'}, inplace=True)
    print("✅ as_ renombrada a away_shots")

# Rename columns to standard ones used in the code
cols_to_rename = {'ftr': 'result', 'fthg': 'home_goals', 'ftag': 'away_goals', 'hy': 'home_yellow_cards'}
matches.rename(columns={k: v for k, v in cols_to_rename.items() if k in matches.columns}, inplace=True)

matches['date'] = pd.to_datetime(matches['date'], format='%d/%m/%Y')
print("✅ Fechas convertidas")

team_map = {
    'Man United': 'Man Utd',
    "Nott'm Forest": 'Nottingham Forest',
    'Spurs': 'Tottenham',
    'Wolves': 'Wolverhampton',
}
for col in ['home_team', 'away_team']:
    if col in matches.columns:
        matches[col] = matches[col].replace(team_map)
print("✅ team_map aplicado")

In [ ]:
events_spatial = events[(events['x'] != 0) | (events['y'] != 0)].copy()
print(f"events original: {len(events)} | events_spatial: {len(events_spatial)}")

In [ ]:
shots = events_spatial[events_spatial['is_shot'] == True].copy()
print(f"Tiros totales: {len(shots)} | Goles: {shots['is_goal'].sum()}")
print(f"Tasa de conversión: {shots['is_goal'].mean()*100:.1f}%")

In [ ]:
if 'qualifiers' in shots.columns:
    q = shots['qualifiers'].astype(str)
else:
    q = pd.Series('', index=shots.index)

shots['is_big_chance']  = q.str.contains('BigChance',  na=False).astype(int)
shots['is_header']      = q.str.contains('"Head"',     na=False).astype(int)
shots['is_right_foot']  = q.str.contains('RightFoot',  na=False).astype(int)
shots['is_left_foot']   = q.str.contains('LeftFoot',   na=False).astype(int)
shots['is_penalty']     = q.str.contains('"Penalty"',  na=False).astype(int)
shots['is_counter']     = q.str.contains('FastBreak',  na=False).astype(int)
shots['from_corner']    = q.str.contains('FromCorner', na=False).astype(int)
shots['is_volley']      = q.str.contains('Volley',     na=False).astype(int)
shots['first_touch']    = q.str.contains('FirstTouch', na=False).astype(int)

qualifier_cols = ['is_big_chance','is_header','is_right_foot','is_left_foot',
                  'is_penalty','is_counter','from_corner','is_volley','first_touch']
print(shots[qualifier_cols].sum().sort_values(ascending=False))

In [ ]:
shots['distance_to_goal'] = np.sqrt((100 - shots['x'])**2 + (50 - shots['y'])**2)
shots['angle_to_goal']    = np.arctan2(np.abs(50 - shots['y']), 100 - shots['x'])
shots['x_norm']           = shots['x'] / 100
shots['y_centered']       = np.abs(shots['y'] - 50) / 50
print(shots[['x','y','distance_to_goal','angle_to_goal','is_goal']].describe())

In [ ]:
shots.to_csv('data/shots_processed.csv', index=False)
matches.to_csv('data/matches_clean.csv', index=False)
print("✅ shots_processed.csv y matches_clean.csv guardados")